# scikit_image

## Example 1: edges of a square

In [ ]:
# Example 1: Canny finds the outline of a bright square; Sobel's response peaks on its border
import numpy as np
from skimage import feature, filters

image = np.zeros((64, 64)); image[16:48, 16:48] = 1.0
edges = feature.canny(image, sigma=1.0)
assert edges.dtype == bool and edges.sum() > 100 and not edges[32, 32] and edges[16:18, 20:44].any()
sobel = filters.sobel(image)
assert sobel[32, 32] == 0 and sobel[32, 16] > 0.4
print("Example 1:", int(edges.sum()), "edge pixels; Sobel max", round(float(sobel.max()), 3))

## Example 2: label blobs and measure them

In [ ]:
# Example 2: measure.label numbers the connected regions; regionprops gives area and centroid
import numpy as np
from skimage import measure

mask = np.zeros((40, 40), dtype=bool)
mask[5:10, 5:10] = True; mask[20:30, 25:35] = True
labels = measure.label(mask)
props = sorted(measure.regionprops(labels), key=lambda p: p.area)
assert labels.max() == 2 and [p.area for p in props] == [25, 100]
assert np.allclose(props[1].centroid, (24.5, 29.5))
print("Example 2:", labels.max(), "regions, areas", [int(p.area) for p in props], "centroid", props[1].centroid)

## Example 3: morphology

In [ ]:
# Example 3: dilation grows a square by the footprint; opening removes a single noisy pixel
import numpy as np
from skimage import morphology

square = np.zeros((20, 20), dtype=bool); square[8:12, 8:12] = True
grown = morphology.dilation(square, morphology.footprint_rectangle((3, 3)))
assert square.sum() == 16 and grown.sum() == 36
noisy = square.copy(); noisy[2, 2] = True
assert not morphology.opening(noisy, morphology.footprint_rectangle((2, 2)))[2, 2]
skeleton = morphology.skeletonize(np.pad(np.ones((5, 30), dtype=bool), 5))
assert 20 <= skeleton.sum() <= 30
print("Example 3: 16 ->", int(grown.sum()), "pixels; skeleton of a bar has", int(skeleton.sum()), "pixels")

## Example 4: geometric transforms

In [ ]:
# Example 4: rotate by 90 degrees matches np.rot90; resize halves the shape
import numpy as np
from skimage import transform

image = np.arange(16, dtype=float).reshape(4, 4)
rotated = transform.rotate(image, 90, order=0, preserve_range=True)
assert np.array_equal(rotated, np.rot90(image))
small = transform.resize(np.ones((64, 32)), (32, 16), anti_aliasing=True)
assert small.shape == (32, 16) and np.allclose(small, 1.0)
tform = transform.SimilarityTransform(scale=2, translation=(1, 1))
assert np.allclose(tform([[1, 1]]), [[3, 3]])
print("Example 4: rot90 ok, resize ->", small.shape, ", similarity maps (1,1) ->", tform([[1, 1]]).tolist())

## Example 5: an OpenMP-parallel Cython routine

In [ ]:
import numpy as np
from skimage import restoration

image = np.full((60, 60), 50.0)
image[25:35, 25:35] += 100.0  # a bright object on a flat background
background = restoration.rolling_ball(image, radius=10, workers=4)
assert background.shape == image.shape and np.allclose(background[5:15, 5:15], 50.0)
assert (image - background)[30, 30] > 90
print("Example 5: background", float(background[10, 10]), "object after subtraction", float((image - background)[30, 30]))